# 04 · Model Training

The core lesson of this project lives here. I train in two ways and show why one of them is honest:

1. **Random 80/20 split** — the standard approach. It looks great (~R² 0.55)… but it is **misleading**.
2. **Walk-forward** — train on the past, predict the future, retrain as new IPOs arrive. This is the
   honest number, and it is the one I report.

Then a quick overfitting check (train vs test gap).


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

PROC = "../data/processed"
target = "listing_gain_close_pct"

df = pd.read_csv(f"{PROC}/ipo_features.csv", parse_dates=["list_date"])
features = [c for c in df.columns if c not in ["company", "year", "list_date", target]]
print(df.shape, "|", len(features), "features")


(309, 24) | 20 features


## 1. The tempting (but misleading) random split

Standard approach: shuffle, take 20% as a test set, try three common models.


In [2]:
X, y = df[features], df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear":           LinearRegression(),
    "RandomForest":     RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(f"{name:18s} R2 {r2_score(y_test, p):.3f}   MAE {mean_absolute_error(y_test, p):.2f}")


Linear             R2 0.370   MAE 14.60


RandomForest       R2 0.558   MAE 13.10


GradientBoosting   R2 0.512   MAE 13.89


RandomForest looks excellent (R² ~0.55). **But this number is not trustworthy.** A random split
mixes IPOs from every year, so a cooled-2026 IPO can sit next to a hot-2023 IPO in training. The model
is partly being tested on a market regime it already saw. Given how sharply the market cooled (the
year chart in notebook 02), the honest test must respect time.


## 2. The honest test — walk-forward

I sort IPOs by listing date, train on everything up to a point, predict the next 5, then slide
forward and retrain including those 5. This mimics real life: it always predicts the future from the
past, and it **keeps retraining** so it adapts to the cooling market.

Three small touches help on only ~300 rows:
- **shallow trees** (depth 1–3) so the model can't memorise,
- **clip the worst 2% of training outliers** so a few crazy IPOs don't distort the fit,
- **average a GradientBoosting and a RandomForest** — their errors partly cancel.


In [3]:
df_time = df.sort_values("list_date").reset_index(drop=True)
start = (df_time["year"] <= 2024).sum()      # start predicting from the first 2025 IPO

def walk_forward(feature_cols):
    preds = []
    i = start
    while i < len(df_time):
        train = df_time.iloc[:i]              # everything before this point
        test  = df_time.iloc[i:i + 5]         # the next 5 IPOs

        lo, hi = train[target].quantile([0.02, 0.98])
        y_clip = train[target].clip(lo, hi)   # clip training outliers

        gb = GradientBoostingRegressor(n_estimators=200, max_depth=1,
                                       learning_rate=0.05, loss="absolute_error", random_state=42)
        rf = RandomForestRegressor(n_estimators=300, max_depth=3, random_state=42)
        gb.fit(train[feature_cols], y_clip)
        rf.fit(train[feature_cols], y_clip)

        batch = (gb.predict(test[feature_cols]) + rf.predict(test[feature_cols])) / 2
        preds.extend(batch)
        i += 5
    return np.array(preds)

actual = df_time.iloc[start:][target].values
full_pred = walk_forward(features)

print("WALK-FORWARD (honest)")
print("  R2 :", round(r2_score(actual, full_pred), 3))
print("  MAE:", round(mean_absolute_error(actual, full_pred), 2))


WALK-FORWARD (honest)
  R2 : 0.499
  MAE: 11.11


On the honest walk-forward test the model scores around **R² 0.50** — a real, leak-free result.
It never sees the future and adapts to the cooling as it retrains. That is the number I would put on a
resume, not the flattering 0.55.


## 3. Is it overfitting?

Compare the score on data the model trained on (train) versus data it never saw (test). A big gap
means memorising; both low means underfitting. I want a small, healthy gap.


In [4]:
train_scores, test_pred, test_true = [], [], []
i = start
while i < len(df_time):
    train = df_time.iloc[:i]
    test  = df_time.iloc[i:i + 5]
    lo, hi = train[target].quantile([0.02, 0.98])
    y_clip = train[target].clip(lo, hi)

    gb = GradientBoostingRegressor(n_estimators=200, max_depth=1,
                                   learning_rate=0.05, loss="absolute_error", random_state=42).fit(train[features], y_clip)
    rf = RandomForestRegressor(n_estimators=300, max_depth=3, random_state=42).fit(train[features], y_clip)

    train_p = (gb.predict(train[features]) + rf.predict(train[features])) / 2
    train_scores.append(r2_score(y_clip, train_p))
    test_pred.extend((gb.predict(test[features]) + rf.predict(test[features])) / 2)
    test_true.extend(test[target].values)
    i += 5

train_r2 = np.mean(train_scores)
test_r2  = r2_score(test_true, test_pred)
print(f"train R2 : {train_r2:.3f}")
print(f"test  R2 : {test_r2:.3f}")
print(f"gap      : {train_r2 - test_r2:.3f}  (small = healthy)")


train R2 : 0.608
test  R2 : 0.499
gap      : 0.109  (small = healthy)


A small gap means the model is neither memorising nor underfitting — the shallow trees are doing
their job on a small dataset. Notebook 05 takes this model design and answers the real question: can
a retail investor act on it, and does it beat just trusting the grey market?
